In [1]:
import oracledb
import pandas as pd
from prediction_all import predict_pKd, predict_pKi, make_smiles, toxic_predict
import torch
import random

oracledb.init_oracle_client(lib_dir=r"D:\\instantclient_23_9")

conn = oracledb.connect(
    user="adsql",          # 사용자명
    password="oracle_4U",      # 비밀번호
    dsn="localhost:1521/xe" # 접속 정보 (SQL Developer와 동일)
)
cur = conn.cursor()

c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'haiku'
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.d

In [3]:
user_selected_button = '암 치료제' # 누른 버튼 이름
sql = f"SELECT * FROM disease_input where d_category = '{user_selected_button}'"
# 암 치료제 등 버튼 누르면 disease_input에서 해당 카테고리의 smiles만 가져옴
res = pd.read_sql(sql, conn)
randoms = [0,1]
orig_molecule = res.iloc[randoms].to_dict(orient='records') # 카테고리별 smiles 2개 관련 정보

# orig_molecule로 새로운 분자 생성 후에 DB에 저장하는 거까지
molecule_id_number = 0
db_col = ['DNEW_CHEMBL_ID', 'DNEW_NAME', 'DNEW_CANOSMILES', 'DNEW_IMAGE_BASE64',
		'DNEW_MOL_WEIGHT', 'DNEW_LOGP', 'DNEW_QED', 'DNEW_HBD', 'DNEW_HBA',
		'DNEW_PKI_RES', 'DNEW_PKI', 'DNEW_PKD_RES', 'DNEW_PKD', 'DNEW_TOXIC',
		'DNEW_CATEGORY']

generate_molecule_result = []

for i in orig_molecule:
	for j in range(5):
		cur.execute("SELECT SEQ_DISEASE_GEN.NEXTVAL FROM DUAL")
		seq_val = cur.fetchone()[0]

		torch.manual_seed(torch.randint(0, 1000000, (1,)).item())
		res = [i['D_CHEMBL_ID']] + [f"DNEW_MOLECULE{seq_val}"] + make_smiles(i['D_CANOSMILES'])
		generate_molecule_result.append(res)
		molecule_id_number += 1
  
for res in generate_molecule_result:
	pki = list(predict_pKi(res[2]))
	pkd = list(predict_pKd(res[2]))
	toxic = [toxic_predict(res[2])]
	res2 = res + pki + pkd + toxic + [i['D_CATEGORY']]
 
	dic = dict(zip(db_col, res2))

	columns = ', '.join(dic.keys())
	placeholders = ', '.join([f':{k}' for k in dic.keys()])

	sql = f"INSERT INTO DISEASE_GENERATIVE ({columns}) VALUES ({placeholders})"

	cur.execute(sql, dic)
	conn.commit()

C:\Users\amysm\AppData\Local\Temp\ipykernel_12088\1206953691.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  res = pd.read_sql(sql, conn)
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Progra